# 04 · Repeated group k-fold by fruit (mean ± SD)  [GPU]
5×3 folds, grouped by physical fruit; per-scan evaluation; mean ± SD across folds.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()/'nbpkg'))
import numpy as np, pandas as pd
from config import CFG
import dataset as ds, eval_core as ec
CFG.out_dir.mkdir(parents=True, exist_ok=True)
print('data_root :', CFG.data_root); print('fruit_key :', CFG.fruit_key)

In [ ]:
import citrus_dl as dl, json
dl.set_seeds(CFG.seed)
fruits=dl.index_fruits(CFG); by={f.fruit_id:f for f in fruits}
ids=np.array([f.fruit_id for f in fruits]); labels=np.array([f.label for f in fruits])
hparams=json.load(open(CFG.out_dir/'hparams.json'))

In [ ]:
rec={bb:[] for bb in CFG.final_backbones}
for rep,tr_ids,te_ids in ec.repeated_group_folds(ids,labels,n_splits=CFG.cv_n_splits,
        n_repeats=CFG.cv_n_repeats,seed=CFG.seed):
    ec.assert_no_leakage(('tr',tr_ids),('te',te_ids))
    tl=[by[i].label for i in tr_ids]
    itr,iva=ec.two_way_split(tr_ids,tl,val_frac=0.2,seed=CFG.seed+rep)
    trf=[by[i] for i in itr]; vaf=[by[i] for i in iva]; tef=[by[i] for i in te_ids]
    for bb in CFG.final_backbones:
        m=dl.train_backbone(CFG,bb,trf,vaf,hparams[bb],verbose=0)
        vs=dl.predict_scan_scores(m,CFG,bb,vaf); ts=dl.predict_scan_scores(m,CFG,bb,tef)
        t,_=ec.select_vote_threshold(vs,objective=CFG.threshold_objective)
        r=ec.evaluate_test(ts,t,with_ci=False)
        rec[bb].append({k:r[k][0] for k in ['AUC','accuracy','precision','recall','F1']})
        import tensorflow as tf; tf.keras.backend.clear_session()
    print('repeat',rep,'done')

In [ ]:
cv=ec.cv_results_to_frame(rec); display(cv)
cv.to_csv(CFG.out_dir/'table3_repeated_cv.csv',index=False)
open(CFG.out_dir/'table3_repeated_cv.md','w').write(cv.to_markdown(index=False))